# Ep. 02 — Los LLM no son magia: tokens, contexto y límites

Curso 1 — Fundamentos de AI Engineering · Linear [CM-42](https://linear.app/caiomedeiros/issue/CM-42)


## Objetivos de aprendizaje

1. Explicar **tokens** y estimar coste desde el tamaño de entrada/salida.
2. Presupuestar una **ventana de contexto finita** (system + historial + docs + respuesta).
3. Internalizar: **probabilidad ≠ verdad** — el muestreo inventa falsedades fluidas.


## Gancho — “¿por qué inventó eso?”

Cuando el modelo inventa una cita, normalmente no está “roto”. Está muestreando los siguientes tokens con contexto finito y evidencia incompleta. Primero sentimos el límite; después lo medimos.


In [ ]:
from __future__ import annotations

import json
import os
import re
import time
from dataclasses import dataclass, field
from typing import Any, Callable

# Optional: set OPENAI_API_KEY to call a real OpenAI-compatible API.
# Without a key, every demo below still runs offline with a tiny mock LLM.
API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
API_BASE = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1").rstrip("/")
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")


def chat(messages: list[dict[str, str]], *, temperature: float = 0.2, max_tokens: int = 300) -> str:
    """Vanilla chat completion: direct HTTP to an OpenAI-compatible API, or offline mock."""
    if not API_KEY:
        return _mock_chat(messages)
    try:
        import urllib.request

        payload = {
            "model": MODEL,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
        }
        req = urllib.request.Request(
            f"{API_BASE}/chat/completions",
            data=json.dumps(payload).encode("utf-8"),
            headers={
                "Authorization": f"Bearer {API_KEY}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        return data["choices"][0]["message"]["content"]
    except Exception as exc:  # noqa: BLE001 — educational fallback
        return f"[API error → offline mock] {exc}\n\n" + _mock_chat(messages)


def _mock_chat(messages: list[dict[str, str]]) -> str:
    """Deterministic offline stand-in so the lesson runs without credentials."""
    user = next((m["content"] for m in reversed(messages) if m["role"] == "user"), "")
    system = next((m["content"] for m in messages if m["role"] == "system"), "")
    lower = user.lower()

    if "json" in system.lower() or "json" in lower:
        return '{"ok": true, "summary": "mock structured answer", "confidence": 0.61}'
    if "capital of france" in lower or "capital de frança" in lower or "capital de francia" in lower:
        return "Paris"
    if "2+2" in lower or "2 + 2" in lower:
        return "4"
    if "invent" in lower or "invente" in lower or "inventa" in lower:
        return "The 1847 Treaty of New Avalon was signed by Admiral Kestrel."
    if "email" in lower or "ssn" in lower or "cpf" in lower:
        return "Sure — contact jane.doe@example.com / SSN 123-45-6789."
    # Default: echo a short, slightly overconfident reply
    snippet = user.strip().replace("\n", " ")[:120]
    return f"Based on my training, the answer is clearly related to: {snippet}…"


print("Mode:", "LIVE API" if API_KEY else "OFFLINE MOCK (set OPENAI_API_KEY for live calls)")
print("Model:", MODEL if API_KEY else "mock-llm")


## Tokenización y coste

Las APIs cobran y truncan en **tokens**, no en caracteres. La celda usa un estimador simple (≈4 chars/token en inglés) y `tiktoken` si está instalado.


In [ ]:
def estimate_tokens(text: str) -> int:
    """Prefer tiktoken when available; otherwise ≈4 characters per token."""
    try:
        import tiktoken  # type: ignore

        enc = tiktoken.get_encoding("cl100k_base")
        return len(enc.encode(text))
    except Exception:
        return max(1, (len(text) + 3) // 4)


def estimate_cost_usd(prompt_tokens: int, completion_tokens: int, *, in_per_m: float = 0.15, out_per_m: float = 0.60) -> float:
    """Default prices resemble a cheap chat model ($/1M tokens) — replace with your provider card."""
    return (prompt_tokens * in_per_m + completion_tokens * out_per_m) / 1_000_000


sample = (
    "AI Engineering turns model capability into reliable product behavior. "
    "Tokens are the unit of context, cost, and truncation."
)
n = estimate_tokens(sample)
print(f"Sample tokens ≈ {n}")
print(f"Cost for 1k similar prompts + 200-token answers ≈ ${estimate_cost_usd(n * 1000, 200 * 1000):.4f}")


## Contexto finito y trade-offs

Todo compite por la misma ventana: system prompt, schemas de tools, historial, chunks recuperados y espacio para la respuesta. Desborde → truncado silencioso o error.


In [ ]:
@dataclass
class ContextBudget:
    window: int = 128_000
    system_tokens: int = 0
    history_tokens: int = 0
    retrieval_tokens: int = 0
    reserve_output: int = 1_000

    def used(self) -> int:
        return self.system_tokens + self.history_tokens + self.retrieval_tokens

    def remaining_for_input(self) -> int:
        return self.window - self.used() - self.reserve_output

    def fit_chunks(self, chunks: list[str]) -> list[str]:
        kept: list[str] = []
        room = self.remaining_for_input()
        for chunk in chunks:
            need = estimate_tokens(chunk) + 4
            if need > room:
                continue  # skip oversized chunk; try smaller later ones
            kept.append(chunk)
            room -= need
            self.retrieval_tokens += need
        return kept


docs = [
    "Doc A: refund policy — 30 days with receipt.",
    "Doc B: " + ("warranty extension details. " * 80),
    "Doc C: shipping SLAs by region.",
    "Doc D: " + ("legacy FAQ noise. " * 120),
]

budget = ContextBudget(window=450, system_tokens=estimate_tokens("You are a support assistant."), history_tokens=80, reserve_output=120)
kept = budget.fit_chunks(docs)
print("Kept chunks:", kept)
print("Retrieval tokens used:", budget.retrieval_tokens)
print("Remaining input room:", budget.remaining_for_input())
print("Limitation first: without budgeting, Doc B/D would crowd out the useful policy text.")


## Probabilidad ≠ verdad

Una respuesta fluida aún puede ser falsa. Ejecuta la demo de muestreo y pide al modelo que invente un hecho histórico — observa confianza sin fundamento.


In [ ]:
import random

def sample_next(candidates: list[tuple[str, float]], temperature: float = 1.0) -> str:
    """Toy sampler: higher temperature flattens probabilities → more surprising (sometimes wrong) tokens."""
    words, weights = zip(*candidates)
    if temperature <= 0:
        return words[weights.index(max(weights))]
    adjusted = [w ** (1.0 / temperature) for w in weights]
    total = sum(adjusted)
    probs = [a / total for a in adjusted]
    return random.choices(words, weights=probs, k=1)[0]


random.seed(7)
cands = [("Paris", 0.62), ("Lyon", 0.18), ("Berlin", 0.12), ("Atlantis", 0.08)]
print("t=0.2 →", [sample_next(cands, 0.2) for _ in range(5)])
print("t=1.5 →", [sample_next(cands, 1.5) for _ in range(5)])

messages = [
    {"role": "system", "content": "Answer briefly. If unsure, invent a confident historical detail."},
    {"role": "user", "content": "Invent the year and signatories of the Treaty of New Avalon."},
]
print("\nModel reply:\n", chat(messages))
print("\nRemember: fluency is not evidence.")


## Trampas + checklist

- Meter PDFs enormes en el prompt “por si acaso”.
- Ignorar el presupuesto de tokens de salida al medir latencia/coste.
- Tratar temperature=0 como “verdad” (sigue siendo predicción del siguiente token).

**Checklist:** contar tokens → reservar respuesta → recuperar solo lo que cabe → citar fuentes.


## Takeaways

1. Los tokens impulsan coste, latencia y lo que el modelo puede ver.
2. El contexto es un recurso escaso — diseña para él.
3. Siguiente: por qué un prompt bonito aún falla sin un sistema.
